# 00 — Colab setup and GPU benchmark (M0)

Sets up the FinCast-LSTM repo on Colab and measures LSTM training speed on this GPU.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 on the free tier).
2. In the Colab **Secrets** panel (key icon), add these and enable *Notebook access* for both:
   - `GH_TOKEN`: a fine-grained GitHub token with **read-only Contents** access to `moghbaie/FinCast-LSTM` (needed to clone the private repo).
   - `TIINGO_API_KEY`: your Tiingo token (used from M1 on).

Data, checkpoints and run outputs are stored in Google Drive under `MyDrive/FinCast-LSTM/`, so they survive disconnects.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FinCast-LSTM'
import os
for d in ['data', 'runs', 'results/benchmarks']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print('Drive root:', DRIVE_ROOT)

Mounted at /content/drive
Drive root: /content/drive/MyDrive/FinCast-LSTM


In [5]:
# Clone or update the private repo. The token is passed via an HTTP header for this call only:
# it is not printed and not stored in .git/config.
import base64, subprocess
from google.colab import userdata
REPO_DIR = '/content/FinCast-LSTM'
URL = 'https://github.com/moghbaie/FinCast-LSTM.git'
tok = userdata.get('GH_TOKEN')
hdr = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
auth = ['git', '-c', f'http.extraHeader={hdr}']
if os.path.exists(f'{REPO_DIR}/.git'):
    r = subprocess.run(auth + ['-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
else:
    r = subprocess.run(auth + ['clone', URL, REPO_DIR], capture_output=True, text=True)
print(r.returncode, r.stdout[-500:], r.stderr[-500:].replace(tok, '***'))
del tok, hdr
!git -C /content/FinCast-LSTM log --oneline -1

0  Cloning into '/content/FinCast-LSTM'...

e788840 (HEAD -> main, origin/main, origin/HEAD) M1: Yahoo vs Tiingo cross-check results


In [6]:
# Keep Colab's preinstalled torch, install the package and its other dependencies.
!pip install -q -e "/content/FinCast-LSTM[data,models,viz,dev]"
# Link data/ and runs/ to Drive so outputs persist.
for d in ['data', 'runs']:
    link = f'{REPO_DIR}/{d}'
    if os.path.islink(link) or not os.path.exists(link):
        if os.path.islink(link): os.unlink(link)
        os.symlink(f'{DRIVE_ROOT}/{d}', link)
%cd /content/FinCast-LSTM

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fincast (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
/content/FinCast-LSTM


In [7]:
# Gate: unit tests (CPU) + GPU test
!python -m pytest -W error -q
!python -m pytest -W error -q -m gpu


==================================== ERRORS ====================================
_____________________ ERROR collecting tests/test_data.py ______________________
tests/test_data.py:8: in <module>
    import pandas as pd
/usr/local/lib/python3.13/dist-packages/pandas/__init__.py:46: in <module>
    from pandas.core.api import (
/usr/local/lib/python3.13/dist-packages/pandas/core/api.py:27: in <module>
    from pandas.core.arrays import Categorical
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/__init__.py:1: in <module>
    from pandas.core.arrays.arrow import ArrowExtensionArray
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/arrow/__init__.py:5: in <module>
    from pandas.core.arrays.arrow.array import ArrowExtensionArray
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/arrow/array.py:97: in <module>
    from pandas.core.arrays.masked import BaseMaskedArray
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: in <module>
    from 

In [12]:
!pip install --force-reinstall /content/FinCast-LSTM
!pip install --upgrade bottleneck
!PYTHONPATH=. python -m pytest -W error -q
!PYTHONPATH=. python -m pytest -W error -q -m gpu

Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pandas-3.1.1-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached pandas_market_calendars-5.6.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached exchange_calendars-4.13.3-py3-none-any.whl.metadata (26 kB)
  Using cached pyluach-2.3.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached toolz-1.2.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached korean_lunar_calendar-0.4.0-py3-none-any.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 32.0 MB/s  0:00:00
Using cached pandas-3.1.1-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.0 MB)
Using cached pandas_market_calendars-5.6.1-py3-none-any.whl (148 kB)
Using cached exchange_calendars-4.13.3-py3-none-any.whl (216 kB)
Using cached korean_lunar_calendar-0.4.0-py3-none-any.whl (11 kB)
   ━━━━━━━━━━━

KeyboardInterrupt: 

In [8]:
# Benchmark (about 5 to 10 min on a T4). Saved to Drive.
import torch, re
gpu = re.sub(r'[^A-Za-z0-9]+', '_', torch.cuda.get_device_name(0)).strip('_').lower()
label = f'colab_{gpu}_v1'
!python scripts/benchmark_lstm.py --label {label} --out {DRIVE_ROOT}/results/benchmarks

{"model": "paper_variational", "batch": 32, "cuda_graph": true, "sec_per_step": 0.02742, "sec_per_epoch_train": 174.8, "sec_val_pass": 0.84, "peak_mem_mb": 367}
{"model": "paper_variational", "batch": 128, "cuda_graph": true, "sec_per_step": 0.02666, "sec_per_epoch_train": 42.5, "sec_val_pass": 0.09, "peak_mem_mb": 383}
{"model": "paper_variational", "batch": 512, "cuda_graph": true, "sec_per_step": 0.03405, "sec_per_epoch_train": 13.6, "sec_val_pass": 0.09, "peak_mem_mb": 400}
{"model": "paper_variational", "batch": 2048, "cuda_graph": true, "sec_per_step": 0.07793, "sec_per_epoch_train": 7.8, "sec_val_pass": 0.1, "peak_mem_mb": 799}
{"model": "cudnn_input_dropout", "batch": 512, "cuda_graph": false, "sec_per_step": 0.0041, "sec_per_epoch_train": 1.6, "sec_val_pass": 0.11, "peak_mem_mb": 424}
wrote /content/drive/MyDrive/FinCast-LSTM/results/benchmarks/colab_tesla_t4_v1.json


**Send the result back:** copy the JSON lines printed above (or the file `MyDrive/FinCast-LSTM/results/benchmarks/colab_*_v1.json`) into the Claude Code chat. They are added to `results/benchmarks/m0_benchmark.md`.